# Fine-tuning

Say you only have a small amount of data.

It can be better to fine-tune a pretrained deep network than to make a shallow network or train a deep network on your data. Of course, it will be better to train on a huge amount of data, but often this is not possible.

The fine-tuned deep network may win because it has already learned spatial feature hierarchy of real-world objects.

In [ ]:
import os
import shutil
import pathlib
import zipfile
import kagglehub

import numpy as np
import tensorflow as tf

import matplotlib.pyplot as plt

from IPython.core.magic import register_cell_magic

os.environ["KERAS_BACKEND"] = "jax"

import keras
import keras_hub


MODELS_DIR = pathlib.Path("models")
MODELS_DIR.mkdir(exist_ok=True)




In [6]:
BATCH_SIZE = 64
IMAGE_SIZE = (180, 180)

DATASET_DIR = pathlib.Path(kagglehub.competition_download("dogs-vs-cats"))
ORIG_TRAIN_DIR = DATASET_DIR / "train"
SMALL_DS_DIR = DATASET_DIR / "dogs_vs_cats_small"

train_dataset = keras.utils.image_dataset_from_directory(
    SMALL_DS_DIR / "train",
    image_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE
)
validation_dataset = keras.utils.image_dataset_from_directory(
    SMALL_DS_DIR / "validation",
    image_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE
)
test_dataset = keras.utils.image_dataset_from_directory(
    SMALL_DS_DIR / "test",
    image_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE
)

conv_base = keras_hub.models.Backbone.from_preset('xception_41_imagenet')
preprocessor = keras_hub.layers.ImageConverter.from_preset(
    "xception_41_imagenet",
    image_size=(180, 180),
)

#  Extracting the Xception features and corresponding labels 
def get_features_and_labels(dataset):
    all_features = []
    all_labels = []
    # LOOP THROUGH THE ENTIRE DATASET
    for images, labels in dataset:
        # preprocess the images
        preprocessed_images = preprocessor(images)
        # run the images through the model
        features = conv_base.predict(preprocessed_images, verbose=0)
        all_features.append(features)
        all_labels.append(labels)
    return np.concatenate(all_features), np.concatenate(all_labels)

# our feature datasets
train_features, train_labels = get_features_and_labels(train_dataset)
val_features, val_labels = get_features_and_labels(validation_dataset)
test_features, test_labels = get_features_and_labels(test_dataset)


# ↓ DEFINITION OF OUR NEW MODEL ---------------------------
inputs = keras.Input(shape=(6, 6, 2048))
x = keras.layers.GlobalAveragePooling2D()(inputs)
x = keras.layers.Dense(256, activation="relu")(x)
x = keras.layers.Dropout(0.25)(x)
outputs = keras.layers.Dense(1, activation="sigmoid")(x)
# ↑ -------------------------------------------------------
model = keras.Model(inputs, outputs)                     # finalise model by giving the inputs & outputs


model.compile(
    loss="binary_crossentropy",
    optimizer="adam",
    metrics=["accuracy"]
)

callbacks = [
    keras.callbacks.ModelCheckpoint(
      filepath = MODELS_DIR / "feature_extraction.keras",
      save_best_only=True,
      monitor="val_loss")
]

history = model.fit(
    train_features,
    train_labels,
    epochs=10,
    validation_data=(val_features, val_labels),
    callbacks=callbacks,
    verbose=0
)

del model

UnauthenticatedError: User is not authenticated

In [5]:
model = keras.models.load_model(MODELS_DIR / "feature_extraction_with_data_augmentation.keras")

ValueError: File not found: filepath=models\feature_extraction_with_data_augmentation.keras. Please ensure the file is an accessible `.keras` zip file.

Giving up, too much setup to do, and I somewhat get the concept